In [1]:
import pandas as pd
import re

# Load dataset
df = pd.read_excel("Journal_Articles.xlsx")  # Replace with your dataset

# Convert to lowercase and remove special characters for authors
def clean_text(text):
    text = str(text).lower().strip()  # Convert to lowercase
    text = re.sub(r'[^a-z\s]', '', text)  # Remove special characters
    return text

df["Standardized_name"] = df["Author_Name"].apply(clean_text)

print(df.head())  # Check cleaned names


      ID                                                URL  \
0  10681  https://www.sciencedirect.com/science/article/...   
1   9394  https://www.sciencedirect.com/science/article/...   
2   2575  https://www.sciencedirect.com/science/article/...   
3   5696  https://www.sciencedirect.com/science/article/...   
4  10114  https://www.sciencedirect.com/science/article/...   

              Journal_Title                  Volume_Issue  month_year  \
0  Decision Support Systems             Volume 3, Issue 2        1987   
1  Decision Support Systems            Volume 26, Issue 2        1999   
2  Decision Support Systems                    Volume 112        2018   
3  Decision Support Systems            Volume 52, Issue 3        2012   
4  Decision Support Systems  Volume 12, Issues 4Ã¢â‚¬â€œ5        1994   

                                            Abstract Keywords  \
0  A message based model of a Distributed Decisio...     "[]"   
1  In theories of strategic management, organizat...

In [2]:
df = df.fillna('NULL')

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert names to TF-IDF vectors (character-based n-grams)
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4))
tfidf_matrix = vectorizer.fit_transform(df["Author_Name"])

print(tfidf_matrix.shape)  # Check matrix size


(10740, 33783)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute similarity scores
similarity_matrix = cosine_similarity(tfidf_matrix)

# Convert to DataFrame
similarity_df = pd.DataFrame(similarity_matrix, index=df["Author_Name"], columns=df["Author_Name"])

print(similarity_df.head())  # Check similarity scores


Author_Name          A Burns  A Rutges  A. Ãƒâ€“zaygen  A. Almansoori  \
Author_Name                                                             
A Burns             1.000000  0.015333        0.000000       0.023178   
A Rutges            0.015333  1.000000        0.014468       0.000000   
A. Ãƒâ€“zaygen      0.000000  0.014468        1.000000       0.035917   
A. Almansoori       0.023178  0.000000        0.035917       1.000000   
LuÃƒÂ­s A. Almeida  0.000000  0.000000        0.053452       0.260357   

Author_Name         LuÃƒÂ­s A. Almeida  A. Bosman  A. Bourouis  A. Dailianas  \
Author_Name                                                                    
A Burns                       0.000000   0.018171     0.037955      0.000000   
A Rutges                      0.000000   0.000000     0.000000      0.000000   
A. Ãƒâ€“zaygen                0.053452   0.044873     0.037174      0.039069   
A. Almansoori                 0.260357   0.089856     0.046649      0.052548   
LuÃƒÂ­s 

In [6]:
# Set similarity threshold (0.75 means 75% similar)
threshold = 0.75  

# Store matched names
matches = {}

for i, name in enumerate(df["Author_University"]):
    for j, other_name in enumerate(df["Author_University"]):
        if i != j and similarity_matrix[i][j] > threshold:
            matches.setdefault(name, []).append(other_name)

# Convert to DataFrame
matches_df = pd.DataFrame(list(matches.items()), columns=["Original Uni", "Similar Uni"])
print(matches_df)


                                           Original Uni  \
0                           Chinese Academy of Sciences   
1                                The University of Iowa   
2                                 University of Bristol   
3                       Queen Mary University of London   
4                             Colorado State University   
...                                                 ...   
1370  Systems Research Institute, Polish Academy of ...   
1371                 Institute for Information Industry   
1372                             Universita' di Bologna   
1373                                   RAND Corporation   
1374                               The Rand Corporation   

                                            Similar Uni  
0     [Chinese Academy of Sciences, Chinese Academy ...  
1     [University of Iowa, University of Iowa, The U...  
2     [University of Bristol, University of Bristol,...  
3     [Queen Mary University of London, Queen Mary U...  
4

In [7]:
import pandas as pd

# Function to get the shortest similar name
def standardize_name(name, matches):
    if name in matches and isinstance(matches[name], list):  # Ensure it's a list
        valid_names = [str(n) for n in matches[name] if pd.notna(n)]  # Convert to strings & remove NaNs
        if valid_names:
            return min(valid_names, key=len)  # Choose the shortest version
    return name  # Keep original if no match

df["Standardized_University"] = df["Author_University"].apply(lambda x: standardize_name(x, matches))
df.to_csv("cleaned_authors.csv", index=False)

print(df[["Author_University", "Standardized_University"]].head())


             Author_University     Standardized_University
0  Chinese Academy of Sciences  Chinese Academy of Science
1  Chinese Academy of Sciences  Chinese Academy of Science
2  Chinese Academy of Sciences  Chinese Academy of Science
3  Chinese Academy of Sciences  Chinese Academy of Science
4       The University of Iowa          University of Iowa
